In [1]:
import torch
import json
import os
import logging
from datetime import datetime
from typing import List, Dict, Any, Optional
from transformers import T5ForConditionalGeneration, T5Tokenizer
import warnings
import re
warnings.filterwarnings('ignore')

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔧 Using device: {device}")

# Main configuration - Updated for FLAN-T5
CONFIG = {
    "model_id": "google/flan-t5-xl",  # or "google/flan-t5-xxl" for better performance
    "use_quantization": True,
    "device": device,
    "max_new_tokens": 512,  # T5 can handle longer outputs
    "temperature": 0.3,
    "input_file": "lulc_extraction_output/output_entities2.json",
    "output_dir": "lulc_extraction_output_flan_t5",
    "max_sentences": 20,
}

# Valid LULC relations
VALID_RELATIONS = [
    "CHANGE_TO", "INCREASES_BY", "DECREASES_BY", "CAUSES", "LOCATED_IN",
    "OCCURS_DURING", "MEASURES", "AFFECTS", "FROM_TO", "ENABLES"
]

# Create output directory
os.makedirs(CONFIG["output_dir"], exist_ok=True)
print(f"✅ Configuration loaded successfully")
print(f"📁 Output directory: {CONFIG['output_dir']}")
print(f"🤖 Model: {CONFIG['model_id']}")

🔧 Using device: cuda
✅ Configuration loaded successfully
📁 Output directory: lulc_extraction_output_flan_t5
🤖 Model: google/flan-t5-xl


Cell 3: Data Loading Function

In [3]:
def load_preprocessed_data(file_path: str) -> List[Dict]:
    """Load already processed data with sentence and entities keys"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        print(f"📥 Loaded {len(data)} items from {file_path}")
        
        processed_data = []
        for item in data:
            # Clean the sentence text
            sentence = item.get('sentence', '')
            if sentence.startswith("text': '"):
                sentence = sentence[7:]
            if sentence.endswith("'"):
                sentence = sentence[:-1]
            
            # Get existing entities (if any)
            entities = item.get('entities', [])
            
            processed_data.append({
                'sentence': sentence,
                'entities': entities,
                'original_data': item
            })
        
        # Print statistics
        total_entities = sum(len(item['entities']) for item in processed_data)
        sentences_with_entities = sum(1 for item in processed_data if item['entities'])
        
        print(f"📊 Processing Statistics:")
        print(f"  - Total sentences: {len(processed_data)}")
        print(f"  - Sentences with entities: {sentences_with_entities}")
        print(f"  - Total entities extracted: {total_entities}")
        if len(processed_data) > 0:
            print(f"  - Average entities per sentence: {total_entities/len(processed_data):.2f}")
        
        return processed_data
        
    except Exception as e:
        logger.error(f"Error loading data: {e}")
        return []

Cell 4: Model Loading Function for FLAN-T5

In [4]:
def load_flan_t5_model(model_id: str, use_quantization: bool = True):
    """Load FLAN-T5 model with proper configuration"""
    print(f"🔄 Loading model: {model_id}")
    
    try:
        # Load tokenizer
        tokenizer = T5Tokenizer.from_pretrained(model_id)
        print(f"📝 Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")
        
        # Load model with appropriate settings
        if use_quantization and device.type == "cuda":
            model = T5ForConditionalGeneration.from_pretrained(
                model_id,
                device_map="auto",
                load_in_8bit=True,  # 8-bit quantization for T5
                torch_dtype=torch.float16
            )
            print("🔧 Using 8-bit quantization")
        else:
            model = T5ForConditionalGeneration.from_pretrained(
                model_id,
                torch_dtype=torch.float32
            )
            if device.type == "cuda":
                model = model.to(device)
        
        print(f"✅ Model loaded successfully")
        return model, tokenizer
        
    except Exception as e:
        logger.error(f"Failed to load model: {e}")
        raise

print("✅ Model loading function defined")

✅ Model loading function defined


Cell 5: Prompt Engineering for FLAN-T5 with CoT

In [85]:
def build_lulc_extraction_prompt_t5(sentence):
    """Build CoT prompt for FLAN-T5 for joint entity recognition and relation extraction"""
    
    prompt=  f"""You are an expert in Land Use and Land Cover (LULC) analysis.

Task: Identify entities and extract relations from the sentence.

Entity types: CHANGE, LOC, LULC, DATE, PERCENT, QUANTITY, COORDINATES, SURFACE_UNIT, PROCESS, CARDINAL, TIME_PERIODS

Relation types: CHANGE_TO, INCREASES_BY, DECREASES_BY, CAUSES, LOCATED_IN, OCCURS_DURING, MEASURES, AFFECTS, FROM_TO, ENABLES

Format:
- ENTITIES: entity_text | ENTITY_TYPE
- RELATIONS: source:ENTITY_TYPE --RELATION-- target:ENTITY_TYPE

Example:
Sentence: Between 1995 and 2005, 30% of Brazil's forest was converted to cropland due to agricultural expansion.

ENTITIES:
- Between 1995 and 2005 | TIME_PERIODS
- 30% | PERCENT
- forest | LULC
- cropland | LULC
- converted | CHANGE
- agricultural expansion | PROCESS
- Brazil | LOC
RELATIONS:
forest:LULC --CHANGE_TO-- cropland:LULC
converted:CHANGE --OCCURS_DURING-- Between 1995 and 2005:TIME_PERIODS
converted:CHANGE --LOCATED_IN-- Brazil:LOC
30%:PERCENT --MEASURES-- converted:CHANGE
agricultural expansion:PROCESS --CAUSES-- converted:CHANGE
Sentence: {sentence}
"""
    return prompt


In [73]:
"""
Example 2:
Sentence: "Urban area expanded from 30 km² in 1990 to 45 km² in 2020 in Shanghai."
Step 1 - Identify entities:
- "Urban area" = LULC
- "expanded" = CHANGE
- "30 km²" = SURFACE_UNIT
- "1990" = DATE
- "45 km²" = SURFACE_UNIT
- "2020" = DATE
- "Shanghai" = LOC

Step 2 - Identify relations:
- Urban area stays urban (same LULC), just grows → NOT CHANGE_TO
- 30 km² changes to 45 km² → FROM_TO
- Both measurements quantify urban area → MEASURES
- Everything happens in Shanghai → LOCATED_IN

ENTITIES:
- Urban area | LULC
- expanded | CHANGE
- 30 km² | SURFACE_UNIT
- 1990 | DATE
- 45 km² | SURFACE_UNIT
- 2020 | DATE
- Shanghai | LOC

RELATIONS:
- Urban area:LULC --LOCATED_IN-- Shanghai:LOC
- 30 km²:SURFACE_UNIT --FROM_TO-- 45 km²:SURFACE_UNIT
- 30 km²:SURFACE_UNIT --MEASURES-- Urban area:LULC
- 45 km²:SURFACE_UNIT --MEASURES-- expanded:CHANGE """


'\nExample 2:\nSentence: "Urban area expanded from 30 km² in 1990 to 45 km² in 2020 in Shanghai."\nStep 1 - Identify entities:\n- "Urban area" = LULC\n- "expanded" = CHANGE\n- "30 km²" = SURFACE_UNIT\n- "1990" = DATE\n- "45 km²" = SURFACE_UNIT\n- "2020" = DATE\n- "Shanghai" = LOC\n\nStep 2 - Identify relations:\n- Urban area stays urban (same LULC), just grows → NOT CHANGE_TO\n- 30 km² changes to 45 km² → FROM_TO\n- Both measurements quantify urban area → MEASURES\n- Everything happens in Shanghai → LOCATED_IN\n\nENTITIES:\n- Urban area | LULC\n- expanded | CHANGE\n- 30 km² | SURFACE_UNIT\n- 1990 | DATE\n- 45 km² | SURFACE_UNIT\n- 2020 | DATE\n- Shanghai | LOC\n\nRELATIONS:\n- Urban area:LULC --LOCATED_IN-- Shanghai:LOC\n- 30 km²:SURFACE_UNIT --FROM_TO-- 45 km²:SURFACE_UNIT\n- 30 km²:SURFACE_UNIT --MEASURES-- Urban area:LULC\n- 45 km²:SURFACE_UNIT --MEASURES-- expanded:CHANGE '

Cell 6: Generation Function for FLAN-T5

In [74]:
def generate_lulc_extraction_t5(sentence, model, tokenizer):
    """Generate entity and relation extraction using FLAN-T5"""
    prompt = build_lulc_extraction_prompt_t5(sentence)
    
    # Tokenize input
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024,  # T5 can handle up to 512-1024 tokens
        padding=True
    )
    
    # Move to device
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # Generate with T5 optimized settings
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=CONFIG["max_new_tokens"],
            temperature=CONFIG["temperature"],
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            num_beams=2,  # Beam search for better quality
            early_stopping=True
        )
    
    # Decode and return
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return response

print("✅ Generation function defined")

✅ Generation function defined


Cell 7: Response Parsing Function

In [75]:
def clean_and_parse_response(response: str, original_sentence: str) -> Dict[str, Any]:
    """Parse FLAN-T5 response - handles both colon and pipe separators"""
    
    result = {
        'entities': [],
        'relations': [],
        'raw_response': response
    }
    
    if not response:
        return result
    
    lines = response.split('\n')
    current_section = None
    
    for line in lines:
        line = line.strip()
        if not line:
            continue
            
        # Check for section transitions
        if "ENTITIES:" in line.upper():
            current_section = "ENTITIES"
            continue
        elif "RELATIONS:" in line.upper():
            current_section = "RELATIONS"
            continue
            
        # Parse entities
        if current_section == "ENTITIES":
            # Remove leading dashes/bullets
            clean_line = line.lstrip('-•· ').strip()
            
            # Try pipe format first: "text | TYPE"
            if '|' in clean_line:
                parts = clean_line.split('|', 1)
                if len(parts) == 2:
                    text = parts[0].strip()
                    entity_type = parts[1].strip()
                    if text and entity_type:
                        result['entities'].append({'text': text, 'type': entity_type})
                        
        # Parse relations
        elif current_section == "RELATIONS":
            if "--" in line:
                relation_text = line.lstrip('-•· ').strip()
                # Validate relation type
                for valid_rel in VALID_RELATIONS:
                    if f"--{valid_rel}--" in relation_text:
                        result['relations'].append(relation_text)
                        break
    
    # Extract missing PERCENT entities
    percentage_pattern = r'\b\d+\.?\d*%\b'
    for relation in result['relations']:
        percentages = re.findall(percentage_pattern, relation)
        for pct in percentages:
            pct_exists = any(entity['text'] == pct for entity in result['entities'])
            if not pct_exists and pct in original_sentence:
                result['entities'].append({'text': pct, 'type': 'PERCENT'})
    
    # Remove duplicates
    seen = set()
    unique_entities = []
    for entity in result['entities']:
        ident = (entity['text'].lower(), entity['type'])
        if ident not in seen:
            seen.add(ident)
            unique_entities.append(entity)
            
    result['entities'] = unique_entities
    return result

print("✅ Parsing function defined")

✅ Parsing function defined


Cell 8: Batch Processing Function

In [76]:
def process_sentences_batch(sentences: List[Dict], model, tokenizer, max_sentences: int = None) -> List[Dict]:
    """Process multiple sentences and extract LULC information"""
    
    if max_sentences:
        sentences = sentences[:max_sentences]
    
    results = []
    
    print(f"🔄 Processing {len(sentences)} sentences...")
    
    for i, item in enumerate(sentences, 1):
        sentence = item['sentence']
        
        if len(sentence) < 10:  # Skip very short sentences
            continue
            
        print(f"📝 Processing {i}/{len(sentences)}: {sentence[:60]}...")
        
        try:
            # Generate extraction
            raw_response = generate_lulc_extraction_t5(sentence, model, tokenizer)
            
            # Parse and clean
            parsed_result = clean_and_parse_response(raw_response, sentence)
            
            # Combine with original data
            result = {
                'sentence': sentence,
                'original_entities': item.get('entities', []),
                'extracted_entities': parsed_result['entities'],
                'extracted_relations': parsed_result['relations'],
                'raw_model_response': parsed_result['raw_response'],
                'processing_timestamp': datetime.now().isoformat()
            }
            
            results.append(result)
            
            # Print progress
            if parsed_result['entities']:
                print(f"    ✓ Found {len(parsed_result['entities'])} entities, {len(parsed_result['relations'])} relations")
            else:
                print(f"    ✗ No entities found")
                
        except Exception as e:
            logger.error(f"Error processing sentence {i}: {e}")
            result = {
                'sentence': sentence,
                'original_entities': item.get('entities', []),
                'extracted_entities': [],
                'extracted_relations': [],
                'error': str(e),
                'processing_timestamp': datetime.now().isoformat()
            }
            results.append(result)
            continue
            
        # Save intermediate results every 10 sentences
        if i % 10 == 0:
            save_results(results, f"{CONFIG['output_dir']}/intermediate_results_{i}.json")
    
    return results

print("✅ Batch processing function defined")

✅ Batch processing function defined


Cell 9: Save Results Function

In [77]:
def save_results(results: List[Dict], output_path: str):
    """Save processing results to JSON file"""
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
        print(f"💾 Saved {len(results)} results to {output_path}")
    except Exception as e:
        logger.error(f"Error saving results: {e}")

print("✅ Save function defined")

✅ Save function defined


Cell 10: Load the Model

In [78]:
# Load the FLAN-T5 model
print("\n🚀 Starting LULC extraction pipeline with FLAN-T5...")
model, tokenizer = load_flan_t5_model(CONFIG["model_id"], CONFIG["use_quantization"])


🚀 Starting LULC extraction pipeline with FLAN-T5...
🔄 Loading model: google/flan-t5-xl


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


📝 Tokenizer loaded. Vocab size: 32000


INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🔧 Using 8-bit quantization
✅ Model loaded successfully


Cell 11: Load Input Data

In [79]:
# Load input data
print("\n📥 Loading input data...")
input_data = load_preprocessed_data(CONFIG["input_file"])


📥 Loading input data...
📥 Loaded 67 items from lulc_extraction_output/output_entities2.json
📊 Processing Statistics:
  - Total sentences: 67
  - Sentences with entities: 67
  - Total entities extracted: 398
  - Average entities per sentence: 5.94


Cell 12: Test with Single Sentence

In [80]:
# Test with single sentence
if input_data:
    print("\n🧪 Testing with single sentence...")
    test_sentence = input_data[2]['sentence']
    print(f"Test sentence: {test_sentence[:150]}...")
    
    # Generate response
    test_response = generate_lulc_extraction_t5(test_sentence, model, tokenizer)
    print(f"\n📤 Raw model response:")
    print(f"'{test_response}'")
    
    # Parse response
    test_parsed = clean_and_parse_response(test_response, test_sentence)
    
    print(f"\n📊 Parsed Results:")
    print(f"   - Entities found: {len(test_parsed['entities'])}")
    for entity in test_parsed['entities']:
        print(f"     • {entity['text']} | {entity['type']}")
    
    print(f"   - Relations found: {len(test_parsed['relations'])}")
    for relation in test_parsed['relations']:
        print(f"     • {relation}")
else:
    print("❌ No data available for testing")


🧪 Testing with single sentence...
Test sentence: )., head: Land use change in the study site, p: ref: 2a, 2b, From 1996 to 2017, the area covered by cropped fields, C, has increased from 40% to 51.3%...

📤 Raw model response:
'CHANGE_TO:51.3% - LULC CHANGE_TO:47% - LULC CHANGE_TO:64% - LULC CHANGE_TO:40% - LULC CHANGE_TO:51.3% - LULC CHANGE_TO:47%'

📊 Parsed Results:
   - Entities found: 0
   - Relations found: 0


In [81]:
# Test with single sentence - Raw response only
if input_data:
    print("\n🧪 Testing with single sentence...")
    test_sentence = input_data[2]['sentence']
    print(f"Test sentence: {test_sentence[:150]}...")
    print(f"Full sentence: {test_sentence}")
    
    # Generate response
    test_response = generate_lulc_extraction_t5(test_sentence, model, tokenizer)
    
    print(f"\n" + "="*80)
    print("📤 RAW MODEL RESPONSE:")
    print("="*80)
    print(test_response)
    print("="*80)
    
    # Optional: Show token information
    prompt = build_lulc_extraction_prompt_t5(test_sentence)
    prompt_tokens = tokenizer.encode(prompt, return_tensors="pt")
    response_tokens = tokenizer.encode(test_response, return_tensors="pt")
    
    print(f"\n📊 Token Information:")
    print(f"   - Input prompt tokens: {prompt_tokens.shape[1]}")
    print(f"   - Output response tokens: {response_tokens.shape[1]}")
    print(f"   - Total tokens used: {prompt_tokens.shape[1] + response_tokens.shape[1]}")
    
else:
    print("❌ No data available for testing")


🧪 Testing with single sentence...
Test sentence: )., head: Land use change in the study site, p: ref: 2a, 2b, From 1996 to 2017, the area covered by cropped fields, C, has increased from 40% to 51.3%...
Full sentence: )., head: Land use change in the study site, p: ref: 2a, 2b, From 1996 to 2017, the area covered by cropped fields, C, has increased from 40% to 51.3% over the study site area, and from 47% to 64% of the arable lands in the study site due to the loss fallows

📤 RAW MODEL RESPONSE:
CHANGE_TO:51.3% - LULC CHANGE_TO:47% - LULC CHANGE_TO:64% - LULC CHANGE_TO:40% - LULC CHANGE_TO:51.3% - LULC CHANGE_TO:47%

📊 Token Information:
   - Input prompt tokens: 454
   - Output response tokens: 76
   - Total tokens used: 530


In [82]:
# Test with single sentence - RAW RESPONSE ONLY
if input_data:
    print("\n🧪 Testing with single sentence...")
    test_sentence = input_data[2]['sentence']
    print(f"Test sentence: {test_sentence[:150]}...")
    
    # Generate response
    test_response = generate_lulc_extraction_t5(test_sentence, model, tokenizer)
    
    print(f"\n📤 Raw model response:")
    print("="*80)
    print(test_response)
    print("="*80)
    
    # Also show the response with visible line breaks
    print(f"\n📋 Response with visible line breaks:")
    print(repr(test_response))
    
    # Show response statistics
    print(f"\n📊 Response Statistics:")
    print(f"   - Total length: {len(test_response)} characters")
    print(f"   - Number of lines: {len(test_response.splitlines())}")
    print(f"   - Contains 'ENTITIES': {'Yes' if 'ENTITIES' in test_response else 'No'}")
    print(f"   - Contains 'RELATIONS': {'Yes' if 'RELATIONS' in test_response else 'No'}")
    
else:
    print("❌ No data available for testing")


🧪 Testing with single sentence...
Test sentence: )., head: Land use change in the study site, p: ref: 2a, 2b, From 1996 to 2017, the area covered by cropped fields, C, has increased from 40% to 51.3%...

📤 Raw model response:
CHANGE_TO:51.3% - LULC CHANGE_TO:47% - LULC CHANGE_TO:64% - LULC CHANGE_TO:40% - LULC CHANGE_TO:51.3% - LULC CHANGE_TO:47%

📋 Response with visible line breaks:
'CHANGE_TO:51.3% - LULC CHANGE_TO:47% - LULC CHANGE_TO:64% - LULC CHANGE_TO:40% - LULC CHANGE_TO:51.3% - LULC CHANGE_TO:47%'

📊 Response Statistics:
   - Total length: 122 characters
   - Number of lines: 1
   - Contains 'ENTITIES': No
   - Contains 'RELATIONS': No


In [83]:
# Test with single sentence - DETAILED RAW RESPONSE
if input_data:
    print("\n🧪 Testing with single sentence...")
    test_sentence = input_data[2]['sentence']
    print(f"\n📝 Input sentence:")
    print(f"'{test_sentence}'")
    print(f"Length: {len(test_sentence)} characters")
    
    # Generate response
    print(f"\n⚡ Generating response...")
    test_response = generate_lulc_extraction_t5(test_sentence, model, tokenizer)
    
    # Display raw response in multiple formats
    print(f"\n📤 RAW MODEL OUTPUT (as string):")
    print("-"*80)
    print(test_response)
    print("-"*80)
    
    print(f"\n🔍 RAW MODEL OUTPUT (with escape characters visible):")
    print("-"*80)
    print(repr(test_response))
    print("-"*80)
    
    print(f"\n📊 Line-by-line breakdown:")
    lines = test_response.splitlines()
    for i, line in enumerate(lines, 1):
        print(f"Line {i}: '{line}'")
    
    print(f"\n📈 Response Analysis:")
    print(f"   - Total characters: {len(test_response)}")
    print(f"   - Total lines: {len(lines)}")
    print(f"   - Non-empty lines: {sum(1 for line in lines if line.strip())}")
    print(f"   - Contains 'ENTITIES': {'✅' if 'ENTITIES' in test_response.upper() else '❌'}")
    print(f"   - Contains 'RELATIONS': {'✅' if 'RELATIONS' in test_response.upper() else '❌'}")
    
    # Check what tokens were generated
    print(f"\n🔢 Token Analysis:")
    response_tokens = tokenizer.encode(test_response, return_tensors="pt")
    print(f"   - Response token count: {response_tokens.shape[1]}")
    
else:
    print("❌ No data available for testing")


🧪 Testing with single sentence...

📝 Input sentence:
')., head: Land use change in the study site, p: ref: 2a, 2b, From 1996 to 2017, the area covered by cropped fields, C, has increased from 40% to 51.3% over the study site area, and from 47% to 64% of the arable lands in the study site due to the loss fallows'
Length: 258 characters

⚡ Generating response...

📤 RAW MODEL OUTPUT (as string):
--------------------------------------------------------------------------------
CHANGE_TO:51.3% - LULC CHANGE_TO:47% - LULC CHANGE_TO:64% - LULC CHANGE_TO:40% - LULC CHANGE_TO:51.3% - LULC CHANGE_TO:47%
--------------------------------------------------------------------------------

🔍 RAW MODEL OUTPUT (with escape characters visible):
--------------------------------------------------------------------------------
'CHANGE_TO:51.3% - LULC CHANGE_TO:47% - LULC CHANGE_TO:64% - LULC CHANGE_TO:40% - LULC CHANGE_TO:51.3% - LULC CHANGE_TO:47%'
----------------------------------------------------------

In [86]:
# Let's diagnose what's going wrong
print("🔍 DIAGNOSIS:\n")

# 1. Check if prompt is too long
test_sentence = input_data[2]['sentence']
full_prompt = build_lulc_extraction_prompt_t5(test_sentence)
prompt_tokens = tokenizer.encode(full_prompt, return_tensors="pt")

print(f"1️⃣ Prompt Length Check:")
print(f"   - Prompt tokens: {prompt_tokens.shape[1]}")
print(f"   - Model max length: 512")
print(f"   - Status: {'❌ TOO LONG' if prompt_tokens.shape[1] > 450 else '✅ OK'}")

# 2. Check what gets truncated
if prompt_tokens.shape[1] > 512:
    truncated = tokenizer.decode(prompt_tokens[0][:512])
    print(f"\n2️⃣ Truncation Issue:")
    print(f"   - Prompt gets cut off at: '...{truncated[-100:]}'")

# 3. Test with minimal prompt
minimal_prompt = f"Extract entities from: '{test_sentence}'"
minimal_tokens = tokenizer.encode(minimal_prompt, return_tensors="pt").to(model.device)

print(f"\n3️⃣ Testing Minimal Prompt:")
print(f"   - Minimal prompt: '{minimal_prompt[:100]}...'")
print(f"   - Tokens: {minimal_tokens.shape[1]}")

# Generate with minimal prompt
with torch.no_grad():
    minimal_output = model.generate(
        minimal_tokens,
        max_new_tokens=100,
        temperature=0.3,
        do_sample=True
    )

minimal_response = tokenizer.decode(minimal_output[0], skip_special_tokens=True)
print(f"   - Minimal response: '{minimal_response}'")

🔍 DIAGNOSIS:

1️⃣ Prompt Length Check:
   - Prompt tokens: 454
   - Model max length: 512
   - Status: ❌ TOO LONG

3️⃣ Testing Minimal Prompt:
   - Minimal prompt: 'Extract entities from: ')., head: Land use change in the study site, p: ref: 2a, 2b, From 1996 to 20...'
   - Tokens: 83
   - Minimal response: ')., head: Land use change in the study site, p: ref: 2a, 2b, From 1996 to 2017, the area covered by cropped fields, C, has increased from 40% to 51.3% over the study site area, and from 47% to 64% of the arable lands in the study site due to the loss fallows'
